<a href="https://colab.research.google.com/github/AndresMontesDeOca/NLP_1/blob/main/Desafios/Desafio_2_AndresMontesDeOca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 49.2 MB/s eta 0:00:00


### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

--2026-07-11 16:47:42--  http://songs_dataset.zip/
Resolving songs_dataset.zip (songs_dataset.zip)... failed: Name or service not known.
wget: unable to resolve host address ‘songs_dataset.zip’
--2026-07-11 16:47:42--  https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip [following]
--2026-07-11 16:47:42--  https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.

In [3]:
# Posibles bandas
os.listdir("./songs_dataset/")

['janisjoplin.txt',
 'radiohead.txt',
 'kanye.txt',
 'lin-manuel-miranda.txt',
 'britney-spears.txt',
 'drake.txt',
 'disney.txt',
 'beatles.txt',
 'bruno-mars.txt',
 'dolly-parton.txt',
 'alicia-keys.txt',
 'blink-182.txt',
 'al-green.txt',
 'ludacris.txt',
 'notorious_big.txt',
 'notorious-big.txt',
 'lorde.txt',
 'kanye-west.txt',
 'bob-dylan.txt',
 'Lil_Wayne.txt',
 'leonard-cohen.txt',
 'nickelback.txt',
 'lady-gaga.txt',
 'eminem.txt',
 'Kanye_West.txt',
 'jimi-hendrix.txt',
 'joni-mitchell.txt',
 'dr-seuss.txt',
 'dj-khaled.txt',
 'nirvana.txt',
 'rihanna.txt',
 'adele.txt',
 'paul-simon.txt',
 'dickinson.txt',
 'bieber.txt',
 'johnny-cash.txt',
 'r-kelly.txt',
 'bjork.txt',
 'bruce-springsteen.txt',
 'amy-winehouse.txt',
 'prince.txt',
 'nursery_rhymes.txt',
 'cake.txt',
 'michael-jackson.txt',
 'bob-marley.txt',
 'patti-smith.txt',
 'nicki-minaj.txt',
 'missy-elliott.txt',
 'lil-wayne.txt']

In [4]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

/tmp/ipykernel_597/3849064916.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [6]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [7]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [8]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [9]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [11]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [12]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [13]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


### 3 - Entrenar embeddings

In [14]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 112816.0859375
Loss after epoch 1: 65959.0703125
Loss after epoch 2: 65933.796875
Loss after epoch 3: 65713.078125
Loss after epoch 4: 63857.59375
Loss after epoch 5: 64131.21875
Loss after epoch 6: 64024.5625
Loss after epoch 7: 64730.15625
Loss after epoch 8: 62517.6875
Loss after epoch 9: 60367.5
Loss after epoch 10: 59770.75
Loss after epoch 11: 58838.0
Loss after epoch 12: 57692.75
Loss after epoch 13: 56462.5
Loss after epoch 14: 55805.6875
Loss after epoch 15: 55859.0
Loss after epoch 16: 51823.3125
Loss after epoch 17: 49910.375
Loss after epoch 18: 49642.0
Loss after epoch 19: 49032.0


(156986, 287740)

### 4 - Ensayar

In [15]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('pretty', 0.898939311504364),
 ('sleep', 0.8652852773666382),
 ('help', 0.8496038317680359),
 ('try', 0.8406051397323608),
 ('cry', 0.8386998176574707),
 ('little', 0.823335587978363),
 ('twist', 0.82172030210495),
 ('seems', 0.8214591145515442),
 ('not', 0.8211495876312256),
 ('high', 0.8155476450920105)]

In [16]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('four', -0.19778630137443542),
 ('five', -0.20902429521083832),
 ('six', -0.2095864713191986),
 ('shake', -0.22284144163131714),
 ('day', -0.2320755124092102),
 ('three', -0.24169862270355225),
 ('our', -0.2603367269039154),
 ('going', -0.26561999320983887),
 ('here', -0.2660878300666809),
 ('two', -0.26648789644241333)]

In [17]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('five', 0.9832456707954407),
 ('three', 0.9756879210472107),
 ('six', 0.9708660244941711),
 ('seven', 0.9614479541778564),
 ('two', 0.9582176804542542),
 ('sixty', 0.9002768397331238),
 ('one', 0.8089418411254883),
 ('crying', 0.8007992506027222),
 ('us', 0.7840763330459595),
 ('fields', 0.7579959630966187)]

In [18]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[("can't", 0.9448155760765076),
 ('buy', 0.936348557472229),
 ('much', 0.9023349285125732),
 ('just', 0.8382388353347778),
 ('that', 0.8346553444862366)]

In [19]:
# Ensayar con una palabra que no está en el vocabulario:
w2v_model.wv.most_similar(negative=["diedaa"])

KeyError: "Key 'diedaa' not present in vocabulary"

In [20]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[-0.26076543 -0.03843755 -0.02569739  0.15053308 -0.01270296 -0.07229752
  0.18759848  0.364075   -0.08411445 -0.2466928   0.09823524  0.05150576
 -0.17174745 -0.02629052 -0.23965557 -0.18261556  0.13862011 -0.02504166
 -0.23345295 -0.18882625  0.00391731  0.23276812 -0.06087385 -0.12987497
  0.05454547 -0.18617211 -0.29992324  0.0654285  -0.06865111 -0.04121192
  0.22484267  0.10209368  0.12724392  0.19074957 -0.16060333  0.33892742
  0.04053574 -0.2972361  -0.15458906 -0.10265355  0.12958445 -0.07908008
  0.04387893  0.03142048  0.1882514   0.3610548  -0.10422956  0.03210551
  0.11634456  0.24899413  0.08381158  0.0117282   0.22049408 -0.01418873
  0.02395703  0.3867995  -0.13965434  0.09790719 -0.20754744  0.05758469
 -0.22668207 -0.12256434  0.14528689 -0.09796016 -0.2618068   0.03961669
  0.14280558  0.0218276  -0.13634472 -0.09357939 -0.21450074 -0.09169442
 -0.11004407  0.10291273 -0.03175396 -0.01506997 -0.17581756  0.12533629
 -0.3026071   0.09099003 -0.07729723 -0.03212855  0

In [21]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 0.9999999403953552),
 ('babe', 0.9117361307144165),
 ('someone', 0.8928177356719971),
 ('need', 0.8833571672439575),
 ('nothing', 0.875154972076416),
 ("didn't", 0.8649984002113342),
 ("there's", 0.8546666502952576),
 ('feed', 0.8394564986228943),
 ('somebody', 0.8394115567207336),
 ('you', 0.8388770818710327)]

In [22]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.9117361307144165),
 ('someone', 0.8928177356719971),
 ('need', 0.8833571672439575),
 ('nothing', 0.8751549124717712),
 ("didn't", 0.8649984002113342),
 ("there's", 0.8546665906906128),
 ('feed', 0.8394564986228943),
 ('somebody', 0.8394115567207336),
 ('you', 0.8388770818710327),
 ('buy', 0.8379150629043579)]

### 5 - Visualizar agrupación de vectores

In [23]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [24]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

In [25]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

In [26]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)